In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection

# ── Fonts: exact match to thesis ─────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "stix",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

# ── Thesis colour palette ─────────────────────────────────────────────────────
WEBGREEN  = (0,   0.5, 0)
WEBBROWN  = (0.6, 0,   0)
ITALIC_RED = (0.75, 0.1, 0.1)

def _pale(rgb, mix=0.82):
    return tuple(c + (1 - c) * mix for c in rgb)

# Path color maps (pale gradient, colour by distance from zero)
CMAPS = {
    0.30: mcolors.LinearSegmentedColormap.from_list(
              "green", [_pale(WEBGREEN, 0.45), _pale(WEBGREEN, 0.88)]),
    0.65: mcolors.LinearSegmentedColormap.from_list(
              "brown", [_pale(WEBBROWN, 0.45), _pale(WEBBROWN, 0.88)]),
}
LINE_COLORS = {0.30: WEBGREEN, 0.65: WEBBROWN}

# ── ERW simulator ─────────────────────────────────────────────────────────────
def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

# ── QSL statistic ─────────────────────────────────────────────────────────────
def qsl_statistic(S, p):
    """
    Q_n^diff = (3-4p) / log(n)  *  sum_{k=2}^{n} (S_k / k)^2
    Returns array of length n+1 (index 0 and 1 are NaN, defined from k=2).
    """
    n = len(S) - 1
    k   = np.arange(1, n + 1, dtype=float)          # k = 1, …, n
    rat = (S[1:] / k) ** 2                           # (S_k / k)^2

    # cumulative sum starting from k=2  (index 1 in rat is k=2)
    cumsum = np.zeros(n + 1)
    cumsum[2:] = np.cumsum(rat[1:])                  # sum_{k=2}^{j}

    with np.errstate(divide='ignore', invalid='ignore'):
        logn       = np.log(np.arange(1, n + 2, dtype=float))  # log(1)…log(n+1)
        logn[0]    = np.nan
        logn[1]    = np.nan
        Q          = (3 - 4 * p) * cumsum / logn
    Q[0] = np.nan
    Q[1] = np.nan
    return Q

# ── Parameters ────────────────────────────────────────────────────────────────
N_STEPS  = 15_000
N_PATHS  = 8
Q_PARAM  = 0.5

PANELS = [
    (0.30, r"$p = 0.30$"),
    (0.65, r"$p = 0.65$"),
]

steps = np.arange(N_STEPS + 1)
rng   = np.random.default_rng()   # unseeded → fresh paths every run

# ── Simulate ──────────────────────────────────────────────────────────────────
all_Q = {}
for p_val, _ in PANELS:
    paths = [simulate_erw(N_STEPS, p=p_val, q=Q_PARAM, rng=rng)
             for _ in range(N_PATHS)]
    all_Q[p_val] = [qsl_statistic(S, p_val) for S in paths]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(5.0, 2.9), sharey=False)
fig.subplots_adjust(left=0.11, right=0.97, bottom=0.18, top=0.88, wspace=0.46)

# start drawing from n=10 to avoid log(n) blow-up near 0
PLOT_FROM = 10

for ax, (p_val, label) in zip(axes, PANELS):
    lcolor = LINE_COLORS[p_val]
    Qs     = all_Q[p_val]

    xs = steps[PLOT_FROM:]

    # ── individual paths (pale) ──────────────────────────────────────────────
    pale = _pale(lcolor, 0.72)
    for Q in Qs:
        ax.plot(xs, Q[PLOT_FROM:],
                color=pale, lw=0.50, alpha=0.80, zorder=2)

    # ── empirical mean (dark) ────────────────────────────────────────────────
    mean_Q = np.nanmean(np.stack([Q[PLOT_FROM:] for Q in Qs], axis=0), axis=0)
    ax.plot(xs, mean_Q,
            color=lcolor, lw=1.1, alpha=0.95, zorder=3)

    # ── limit line at 1 ─────────────────────────────────────────────────────
    ax.axhline(1.0, color="black", lw=0.75, ls=(0, (5, 3)),
               alpha=0.55, zorder=4)

    # ── axes cosmetics ───────────────────────────────────────────────────────
    ax.set_title(label, fontsize=10, pad=5)
    ax.set_xlabel(r"$n$", labelpad=3)

    ax.set_xlim(PLOT_FROM, N_STEPS)

    # y-limits: give some breathing room around [0, max_path]
    all_vals = np.concatenate([Q[PLOT_FROM:] for Q in Qs])
    finite   = all_vals[np.isfinite(all_vals)]
    ylo      = max(0.0, np.percentile(finite, 1)  - 0.10)
    yhi      =         np.percentile(finite, 99) + 0.15
    ax.set_ylim(ylo, yhi)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(
        ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"${int(x)}$"))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))

    if ax is axes[0]:
        ax.set_ylabel(r"$Q_n^{\,\mathrm{diff}}$", labelpad=4)


plt.show()
